# H1: Did mean arrival delay at SEA change between summer 2023 and summer 2024?

Pre-registered in `docs/ANALYSIS_PLAN.md`: completed flights arriving at Seattle-Tacoma, June 1 to August 31 of each year. H0: equal mean arrival delay. Primary test Welch's t; robustness: Mann-Whitney U, a bootstrapped CI of the mean difference (10,000 resamples), and Welch's t on daily means (within-day clustering). Practical threshold: 2 minutes.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display
from scipy import stats

from delaystats import plotting, results
from delaystats import stats_helpers as sh
from delaystats.athena import run_query

ALPHA = 0.05  # pre-registered, two-sided (docs/ANALYSIS_PLAN.md)
SEED = 20261014  # fixed for every bootstrap
pd.set_option("display.float_format", "{:,.4f}".format)


def fmt_p(p):
    return "< 0.0001" if p < 1e-4 else f"{p:.4f}"


def table(rs):
    return pd.DataFrame([r.as_row() for r in rs]).set_index("test")


THRESHOLD = 2.0  # minutes of mean arrival delay

## Data

In [ ]:
flights = run_query("h1_sea_summer_flights")
a = flights.loc[flights["year"] == 2024, "arrdelay"].to_numpy(float)  # summer 2024
b = flights.loc[flights["year"] == 2023, "arrdelay"].to_numpy(float)  # summer 2023
groups = ("summer_2024", "summer_2023")
print(f"summer 2024: {len(a):,} flights, mean {a.mean():.2f} min, sd {a.std(ddof=1):.2f}")
print(f"summer 2023: {len(b):,} flights, mean {b.mean():.2f} min, sd {b.std(ddof=1):.2f}")
cancellations = run_query("h1_sea_summer_cancellations")
cancellations

## Assumption checks

Q-Q plots per group and Levene's test (reported, not used to gate the test choice: Welch's t does not assume equal variances).

In [ ]:
plotting.qq_plots(
    {"Summer 2023": b, "Summer 2024": a},
    "h1_qq_plots",
    "H1: arrival delays against a normal distribution",
)
plotting.group_distributions(
    {"Summer 2023": b, "Summer 2024": a},
    "h1_distributions",
    "H1: arrival delay at SEA, summer 2023 vs 2024 (dashed: means)",
    "Arrival delay (minutes, clipped to -60..180 for display)",
)
levene = stats.levene(a, b, center="median")
print(f"Levene (median-centred): W = {levene.statistic:.2f}, p = {fmt_p(levene.pvalue)}")
print(f"variance ratio 2024/2023: {a.var(ddof=1) / b.var(ddof=1):.3f}")

## Primary test and robustness checks

In [ ]:
welch = sh.welch_t(a, b, name="h1_welch_flight_level", groups=groups)
mwu = sh.mann_whitney(a, b, name="h1_mann_whitney", groups=groups, n_boot=2000, seed=SEED)
boot = sh.bootstrap_mean_diff(
    a, b, name="h1_bootstrap_mean_diff", groups=groups, n_boot=10_000, seed=SEED
)

daily = flights.groupby(["year", "flightdate"], as_index=False)["arrdelay"].mean()
day_a = daily.loc[daily["year"] == 2024, "arrdelay"].to_numpy(float)
day_b = daily.loc[daily["year"] == 2023, "arrdelay"].to_numpy(float)
welch_day = sh.welch_t(day_a, day_b, name="h1_welch_day_level", groups=("days_2024", "days_2023"))

h1 = [welch, welch_day, mwu, boot]
results.save("H1", h1)
table(h1)

## Interpretation

In [ ]:
def verdict(ci, t=THRESHOLD):
    low, high = ci
    if low > t or high < -t:
        return f"lies entirely beyond the {t:g}-minute practical threshold"
    if -t <= low and high <= t:
        return f"lies entirely within ±{t:g} minutes, so the change is operationally negligible"
    return (
        f"straddles the {t:g}-minute practical threshold, so a meaningful change is not established"
    )


diff = welch.estimate
direction = "rose" if diff > 0 else "fell"
sig_flight = welch.p <= ALPHA
sig_day = welch_day.p <= ALPHA
c24, c23 = cancellations.set_index("year").loc[2024], cancellations.set_index("year").loc[2023]
display(
    Markdown(f"""
Mean arrival delay at SEA {direction} by **{abs(diff):.2f} minutes** from summer 2023 to summer 2024
(Welch's t({welch.df:.0f}) = {welch.statistic:.2f}, p = {fmt_p(welch.p)}, Hedges' g = {welch.effect_size:.3f},
95% CI of the difference [{welch.estimate_ci[0]:.2f}, {welch.estimate_ci[1]:.2f}] minutes;
n = {len(b):,} and {len(a):,} flights).

- **Statistical significance** (alpha = {ALPHA}, two-sided, a single pre-registered test so no
  multiplicity correction): flight level {"significant" if sig_flight else "not significant"};
  day level {"significant" if sig_day else "not significant"} (Welch's t({welch_day.df:.0f}) =
  {welch_day.statistic:.2f}, p = {fmt_p(welch_day.p)}, on {len(day_b)} and {len(day_a)} daily means).
- **Practical significance:** the flight-level CI {verdict(welch.estimate_ci)}; the day-level CI
  [{welch_day.estimate_ci[0]:.2f}, {welch_day.estimate_ci[1]:.2f}] {verdict(welch_day.estimate_ci)};
  the bootstrap CI [{boot.effect_ci[0]:.2f}, {boot.effect_ci[1]:.2f}] {verdict(boot.effect_ci)}.
- **Which result to trust:** flights on the same day share weather and congestion, so the
  flight-level test overstates certainty. The day-level result is the honest one.
- **Distribution shift:** Mann-Whitney U (p = {fmt_p(mwu.p)}, rank-biserial {mwu.effect_size:.3f})
  tests whether one summer's delays tend to be larger, not whether the means differ; with heavy
  right tails, the two can disagree.
- **Service context:** cancellations were {c23.cancellation_rate:.2%} of scheduled SEA arrivals in summer
  2023 and {c24.cancellation_rate:.2%} in 2024. Cancelled flights are excluded from delay averages, so a
  rise in cancellations can make delays look better while service gets worse.
""")
)